# EEG Cohort & Extraction Audit

**Term Project — EEG data availability and extraction QC**

This notebook audits exported EEG band-power CSVs and participant processing summaries. It **does not** reprocess raw EEG, assess raw-signal quality, or perform inferential statistics.

### Audits performed
- Participant, session, task-condition coverage
- Expected channels and frequency bands
- Expected rows per recording (32 channels × 5 bands = 160)
- Missing absolute/relative power values
- Duplicate extraction keys
- Processing-status summaries and recorded skip reasons

**Inputs:** the eight EEG result/processing-summary CSVs. The notebook searches a small set of likely project locations.

**Outputs:** audit CSVs are written to a dedicated audit-output folder (created automatically), rather than mixed with the input files.

In [1]:
from pathlib import Path
import pandas as pd

FILES = {
    "Math Control": ("eeg_analysis_results_math_control.csv", "subject_processing_summary_math_control.csv"),
    "Math Experiment": ("eeg_analysis_results_math_experiment.csv", "subject_processing_summary_math_experiment.csv"),
    "Reading Control": ("eeg_analysis_results_read_control.csv", "subject_processing_summary_read_control.csv"),
    "Reading Experiment": ("eeg_analysis_results_read_experiment.csv", "subject_processing_summary_read_experiment.csv"),
}

EXPECTED_CHANNELS = 32
EXPECTED_BANDS = 5
EXPECTED_POWER_ROWS_PER_RECORDING = EXPECTED_CHANNELS * EXPECTED_BANDS
EXPECTED_SESSIONS = ["ses-1", "ses-2"]
EXPECTED_MATH_CONDITIONS = ["Baseline", "Activity1", "Activity2"]
EXPECTED_READING_CONDITIONS = ["Baseline", "Activity"]

# Candidate locations support common project layouts:
# notebook folder, its parent, root Results/, analysis/results/eeg/, and /mnt/data.
candidate_dirs = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "Results",
    Path.cwd().parent / "Results",
    Path.cwd() / "analysis" / "results" / "eeg",
    Path.cwd().parent / "analysis" / "results" / "eeg",
    Path("/mnt/data"),
]
DATA_DIR = next(
    (d for d in candidate_dirs if d.exists() and (d / FILES["Math Control"][0]).exists()),
    None
)
if DATA_DIR is None:
    checked = "\\n".join(str(d.resolve()) for d in candidate_dirs)
    raise FileNotFoundError(
        "Could not locate the EEG input CSVs. Checked these folders:\\n" + checked
    )

# Keep audit outputs separate from source files.
OUTPUT_DIR = Path.cwd() / "outputs" / "eeg_cohort_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input folder:  {DATA_DIR.resolve()}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

Input folder:  /Users/ananyosen/Documents/Term Project/Results
Output folder: /Users/ananyosen/Documents/Term Project/analysis/outputs/eeg_cohort_audit


In [2]:
# Confirm all expected files are available before loading.
missing_inputs = [fname for pair in FILES.values() for fname in pair if not (DATA_DIR / fname).exists()]
if missing_inputs:
    raise FileNotFoundError("Missing input files:\\n" + "\\n".join(missing_inputs))
print(f"All {sum(len(pair) for pair in FILES.values())} input files found.")

All 8 input files found.


In [3]:
# Load result files and processing summaries, with basic schema checks.
loaded = {}
required_result_columns = {"subject_id", "session", "run", "condition", "channel", "band"}
required_summary_columns = {"subject_id", "status"}

for task_group, (result_name, summary_name) in FILES.items():
    results = pd.read_csv(DATA_DIR / result_name)
    summary = pd.read_csv(DATA_DIR / summary_name)

    missing_result_cols = required_result_columns - set(results.columns)
    missing_summary_cols = required_summary_columns - set(summary.columns)
    if missing_result_cols:
        raise ValueError(f"{task_group} result file missing columns: {sorted(missing_result_cols)}")
    if missing_summary_cols:
        raise ValueError(f"{task_group} processing summary missing columns: {sorted(missing_summary_cols)}")

    loaded[task_group] = {"results": results, "summary": summary}
    print(
        f"{task_group}: {len(results):,} power rows; "
        f"{summary['subject_id'].nunique()} participant IDs in processing summary"
    )

Math Control: 14,400 power rows; 22 participant IDs in processing summary
Math Experiment: 18,240 power rows; 19 participant IDs in processing summary
Reading Control: 7,680 power rows; 16 participant IDs in processing summary
Reading Experiment: 10,880 power rows; 19 participant IDs in processing summary


In [4]:
# Run the cohort and extraction-quality audit.
subject_rows = []
taskgroup_rows = []
missing_rows = []
duplicate_rows = []
condition_rows = []
summary_status_rows = []

for task_group, bundle in loaded.items():
    df = bundle["results"]
    summ = bundle["summary"]
    task, group = task_group.split(" ", 1)
    extracted_ids = set(df["subject_id"].dropna().unique())
    summary_extracted_ids = set(summ.loc[summ["status"].eq("EXTRACTED"), "subject_id"])
    all_ids = sorted(set(summ["subject_id"].dropna()) | extracted_ids)

    expected_conditions = EXPECTED_MATH_CONDITIONS if task == "Math" else EXPECTED_READING_CONDITIONS
    expected_key = ["subject_id", "session", "run", "condition", "channel", "band"]
    duplicate_count = int(df.duplicated(expected_key, keep=False).sum())
    duplicate_rows.append({
        "task": task, "group": group, "result_file": FILES[task_group][0],
        "duplicate_key_rows": duplicate_count, "duplicate_key_definition": "|".join(expected_key)
    })

    for _, srow in summ.iterrows():
        summary_status_rows.append({
            "task": task, "group": group, "subject_id": srow["subject_id"],
            "processing_status": srow["status"],
            "recordings_loaded": srow.get("recordings_loaded"),
            "band_power_rows_summary": srow.get("band_power_rows"),
            "events_files_complete": all(bool(srow.get(col, False)) for col in summ.columns if col.endswith("_events_file")),
            "reason": srow.get("missing_or_inconsistent_reason")
        })

    for sid in all_ids:
        sub = df[df["subject_id"].eq(sid)]
        summary_row = summ[summ["subject_id"].eq(sid)]
        status = summary_row["status"].iloc[0] if len(summary_row) else "NOT_IN_SUMMARY"
        present_sessions = sorted(sub["session"].dropna().unique())
        present_conditions = sorted(sub["condition"].dropna().unique())

        for session in EXPECTED_SESSIONS:
            for condition in expected_conditions:
                block = sub[sub["session"].eq(session) & sub["condition"].eq(condition)]
                if block.empty:
                    missing_rows.append({
                        "task": task, "group": group, "subject_id": sid,
                        "missing_session_condition": f"{session}:{condition}"
                    })
                    continue

                n_channels = block["channel"].nunique()
                n_bands = block["band"].nunique()
                row_count = len(block)
                missing_power_cols = [c for c in block.columns if c.lower() in {"absolute_power", "relative_power", "abs_power", "rel_power"}]
                missing_power_values = int(block[missing_power_cols].isna().sum().sum()) if missing_power_cols else None
                condition_rows.append({
                    "task": task, "group": group, "subject_id": sid,
                    "session": session, "condition": condition,
                    "rows": row_count, "unique_channels": n_channels, "unique_bands": n_bands,
                    "expected_rows": EXPECTED_POWER_ROWS_PER_RECORDING,
                    "rows_match_expected": row_count == EXPECTED_POWER_ROWS_PER_RECORDING,
                    "channels_match_expected": n_channels == EXPECTED_CHANNELS,
                    "bands_match_expected": n_bands == EXPECTED_BANDS,
                    "power_columns_checked": "|".join(missing_power_cols),
                    "missing_power_values": missing_power_values
                })

        subject_rows.append({
            "task": task, "group": group, "subject_id": sid,
            "processing_status": status,
            "has_extracted_data": sid in extracted_ids,
            "summary_says_extracted": sid in summary_extracted_ids,
            "sessions_present": "|".join(present_sessions),
            "conditions_present": "|".join(present_conditions),
            "power_rows": len(sub)
        })

    taskgroup_rows.append({
        "task": task, "group": group,
        "participants_in_summary": summ["subject_id"].nunique(),
        "participants_with_data": len(extracted_ids),
        "participants_marked_extracted": len(summary_extracted_ids),
        "power_rows": len(df),
        "missing_condition_blocks": sum(1 for r in missing_rows if r["task"] == task and r["group"] == group)
    })

print("Audit computation complete.")

Audit computation complete.


In [5]:
# Save audit tables to the dedicated output directory.
outputs = {
    "eeg_cohort_audit.csv": pd.DataFrame(subject_rows),
    "eeg_cohort_summary.csv": pd.DataFrame(taskgroup_rows),
    "eeg_missing_data_audit.csv": pd.DataFrame(missing_rows, columns=["task", "group", "subject_id", "missing_session_condition"]),
    "eeg_duplicate_key_audit.csv": pd.DataFrame(duplicate_rows),
    "eeg_condition_audit.csv": pd.DataFrame(condition_rows),
    "eeg_processing_status_audit.csv": pd.DataFrame(summary_status_rows),
}

for filename, table in outputs.items():
    destination = OUTPUT_DIR / filename
    table.to_csv(destination, index=False)
    print(f"Saved: {destination} ({len(table):,} rows)")

Saved: /Users/ananyosen/Documents/Term Project/analysis/outputs/eeg_cohort_audit/eeg_cohort_audit.csv (76 rows)
Saved: /Users/ananyosen/Documents/Term Project/analysis/outputs/eeg_cohort_audit/eeg_cohort_summary.csv (4 rows)
Saved: /Users/ananyosen/Documents/Term Project/analysis/outputs/eeg_cohort_audit/eeg_missing_data_audit.csv (66 rows)
Saved: /Users/ananyosen/Documents/Term Project/analysis/outputs/eeg_cohort_audit/eeg_duplicate_key_audit.csv (4 rows)
Saved: /Users/ananyosen/Documents/Term Project/analysis/outputs/eeg_cohort_audit/eeg_condition_audit.csv (320 rows)
Saved: /Users/ananyosen/Documents/Term Project/analysis/outputs/eeg_cohort_audit/eeg_processing_status_audit.csv (76 rows)


In [6]:
# Display key cohort summary and QC indicators.
display(outputs["eeg_cohort_summary.csv"])

print("\\nDuplicate-key audit:")
display(outputs["eeg_duplicate_key_audit.csv"])

print("\\nMissing-data audit (first 25 rows):")
display(outputs["eeg_missing_data_audit.csv"].head(25))

print("\\nCondition-level QC flags (first 25 rows):")
display(outputs["eeg_condition_audit.csv"].head(25))

,task,group,participants_in_summary,participants_with_data,participants_marked_extracted,power_rows,missing_condition_blocks
0,Math,Control,22,15,15,14400,42
1,Math,Experiment,19,19,19,18240,0
2,Reading,Control,16,12,12,7680,16
3,Reading,Experiment,19,17,17,10880,8


\nDuplicate-key audit:


,task,group,result_file,duplicate_key_rows,duplicate_key_definition
0,Math,Control,eeg_analysis_results_math_control.csv,0,subject_id|session|run|condition|channel|band
1,Math,Experiment,eeg_analysis_results_math_experiment.csv,0,subject_id|session|run|condition|channel|band
2,Reading,Control,eeg_analysis_results_read_control.csv,0,subject_id|session|run|condition|channel|band
3,Reading,Experiment,eeg_analysis_results_read_experiment.csv,0,subject_id|session|run|condition|channel|band


\nMissing-data audit (first 25 rows):


,task,group,subject_id,missing_session_condition
0,Math,Control,Mc01,ses-1:Baseline
1,Math,Control,Mc01,ses-1:Activity1
2,Math,Control,Mc01,ses-1:Activity2
3,Math,Control,Mc01,ses-2:Baseline
4,Math,Control,Mc01,ses-2:Activity1
5,Math,Control,Mc01,ses-2:Activity2
6,Math,Control,Mc08,ses-1:Baseline
7,Math,Control,Mc08,ses-1:Activity1
8,Math,Control,Mc08,ses-1:Activity2
9,Math,Control,Mc08,ses-2:Baseline


\nCondition-level QC flags (first 25 rows):


,task,group,subject_id,session,condition,rows,unique_channels,unique_bands,expected_rows,rows_match_expected,channels_match_expected,bands_match_expected,power_columns_checked,missing_power_values
0,Math,Control,Mc02,ses-1,Baseline,160,32,5,160,True,True,True,absolute_power|relative_power,0
1,Math,Control,Mc02,ses-1,Activity1,160,32,5,160,True,True,True,absolute_power|relative_power,0
2,Math,Control,Mc02,ses-1,Activity2,160,32,5,160,True,True,True,absolute_power|relative_power,0
3,Math,Control,Mc02,ses-2,Baseline,160,32,5,160,True,True,True,absolute_power|relative_power,0
4,Math,Control,Mc02,ses-2,Activity1,160,32,5,160,True,True,True,absolute_power|relative_power,0
5,Math,Control,Mc02,ses-2,Activity2,160,32,5,160,True,True,True,absolute_power|relative_power,0
6,Math,Control,Mc03,ses-1,Baseline,160,32,5,160,True,True,True,absolute_power|relative_power,0
7,Math,Control,Mc03,ses-1,Activity1,160,32,5,160,True,True,True,absolute_power|relative_power,0
8,Math,Control,Mc03,ses-1,Activity2,160,32,5,160,True,True,True,absolute_power|relative_power,0
9,Math,Control,Mc03,ses-2,Baseline,160,32,5,160,True,True,True,absolute_power|relative_power,0


## Interpretation and scope

- This audit checks exported band-power data structure and availability; it is **not** a raw EEG signal-quality assessment.
- A participant missing a task/session block should not automatically be excluded from all analyses. Inclusion should be comparison- and outcome-specific.
- The notebook performs no hypothesis tests and does not establish EEG effects.
- The next EEG step is to define participant-level aggregation and primary outcomes before inferential analysis.
- Keep source CSVs unchanged; audit outputs are written separately under `outputs/eeg_cohort_audit/`.